# Accuracy alert

Fails the job when recent scorecard pass rate drops below 95 percent. Used as a scheduled gate, not as a silent dashboard.


## Bind config


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Check pass rate


In [ ]:
tables = {r.tableName for r in spark.sql(f"SHOW TABLES IN {cfg.fqn}").collect()}
if "scorecard_runs" not in tables:
    print("SKIP no scorecard_runs table yet")
else:
    row = spark.sql(
        f"""SELECT COUNT_IF(verdict='PASS') AS passes, COUNT(*) AS n FROM {cfg.table('scorecard_runs')}"""
    ).collect()[0]
    rate = (row["passes"] / row["n"]) if row["n"] else 0
    print("pass_rate", rate, "n", row["n"])
    assert rate >= 0.95, f"accuracy {rate} below 0.95"
    print("PASS accuracy gate")
